In [75]:
# Development settings
# ============================================================
%load_ext autoreload
%autoreload 2

# ============================================================
# Standard library
# ============================================================
import sys
from pathlib import Path
import matplotlib.colors as mcolors
import colorcet as cc
import shutil
# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# Project modules
# ============================================================
from src.mapping_process import *
from src.plotting_function import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "figures"

print("Project root:", PROJECT_ROOT)
print("Data:", DATA_DIR)
print("Figures:", FIGURE_DIR)

Project root: f:\TUD_CEG_Repository\MSc_Thesis\coastal-multi-hazard-amplification
Data: f:\TUD_CEG_Repository\MSc_Thesis\coastal-multi-hazard-amplification\data
Figures: f:\TUD_CEG_Repository\MSc_Thesis\coastal-multi-hazard-amplification\figures


## Initialize the Global Coastal Transect System

Load the Global Coastal Transect System (GCTS) including Global Coastal Transect Repository (GCTR), which provides the common spatial framework used to integrate the individual coastal hazard datasets and the required data, such as shoreline change rate. Access for the GCTR required key code granted by Deltares. Please specify the `env_path` with valid key.



In [ ]:
# Configure cloud and Dask settings
env_path = r'f:\TUD_CEG_Repository\Internship\Code\ATT64895.env'
columns = ["transect_id", 'bearing', 'utm_epsg', "geometry", "sds:change_rate",
    'buildings:dist_to_shoreline','class:is_built_environment',
    'continent', 'common_country_name']


gdf = initialize_gcts(env_path,columns)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Map Land Subsidence to Coastal Transects

Associate the global land-subsidence dataset with the coastal transect system. The subsidence information is provided as a raster and is therefore spatially sampled to obtain a representative value for each coastal transect.

For each transect, the nearest valid raster pixel is identified and its subsidence value is assigned to the transect. A maximum search distance is applied to prevent values from distant valid pixels being assigned across large data gaps. Transects without a valid subsidence observation within this distance remain unassigned.

This step converts the raster-based subsidence information into transect-level attributes that can subsequently be combined with the other hazard indicators.


In [ ]:
raster_path = DATA_DIR / 'Subsidence/Final_subsidence_prediction.tif'
gdf = sampling_nearest_pixel_to_gcts(gdf, raster_path,max_dist=0.05)

## Map Coastal Flood Hazard to Coastal Transects

Sample the global coastal flood-depth dataset along each coastal transect. In contrast to the subsidence dataset, which is assigned using the nearest valid raster pixel, flood depth is sampled along the transect because inundation conditions can vary across its shore-normal profile.

Raster values intersecting each transect are sampled at regular intervals and summarized into transect-level flood statistics. The resulting statistics characterize the flood conditions represented by the global flood map along each coastal sampling unit.

Because the global transect dataset contains millions of features, the processing is performed in batches. Each batch is temporarily written to disk before the results are recombined, reducing the memory required during raster sampling.

In [ ]:
raster_path = r'f:\TUD_CEG_Repository\MSc_Thesis\coastal-multi-hazard-amplification\data\Floodmap_noveg_rp100\coast_inun_depth_noveg.vrt'
gdf = sampling_intersect_pixel_to_gcts(gdf,raster_path,
                                     prefix = 'flood',
                                     batch_size = 100_000,
                                     data_dir = DATA_DIR)

## Save the Integrated Hazard Dataset

After all hazard datasets have been mapped to the common transect system, the resulting GeoDataFrame can be stored as a Parquet file for subsequent multi-hazard analysis.

The saved dataset contains the original transect information together with the mapped shoreline-change, land-subsidence, and coastal-flooding attributes. This file therefore represents the integrated hazard dataset used as input for the subsequent hazard scoring, multi-hazard burden, amplification, and hotspot-identification analyses.

The export command is intentionally disabled by default to prevent an existing processed dataset from being overwritten accidentally.

In [ ]:
# Uncomment only when the existing mapped hazard dataset should be replaced
# gdf.to_parquet(DATA_DIR / 'mapped_hazard.parquet')